## Setup & DagHub Connection

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")

# Kaggle secrets → env vars (works when "notebook access" is toggled on)
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for _k in ("DAGSHUB_TOKEN", "DAGSHUB_USERNAME", "DAGSHUB_REPO"):
        try:
            os.environ[_k] = _s.get_secret(_k)
            
        except Exception:
            pass
except ImportError:
    pass  # not on Kaggle — rely on env vars already set

import dagshub, mlflow

DAGSHUB_TOKEN    = os.environ["DAGSHUB_TOKEN"]
DAGSHUB_USERNAME = os.environ["DAGSHUB_USERNAME"]
DAGSHUB_REPO     = os.environ.get("DAGSHUB_REPO", "fraud_detection_kaggle")

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN

dagshub.auth.add_app_token(token=DAGSHUB_TOKEN)
dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=DAGSHUB_REPO, mlflow=True)
print("MLflow tracking URI:", mlflow.get_tracking_uri())


## Best Model-ის ჩატვირთვა Registry-დან

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np

REGISTERED_MODEL = "IEEE_Fraud_BestModel"

# ვტვირთავთ უახლეს ვერსიას
client = mlflow.MlflowClient()
versions = client.search_model_versions(f"name='{REGISTERED_MODEL}'")
latest = max(versions, key=lambda v: int(v.version))
print(f"Loading {REGISTERED_MODEL} v{latest.version} (run_id={latest.run_id})")

model_uri = f"models:/{REGISTERED_MODEL}/{latest.version}"
pipeline  = mlflow.sklearn.load_model(model_uri)
print("Pipeline-ი ჩაიტვირთა:", type(pipeline).__name__)
print("Pipeline-ის ნაბიჯები:", [s[0] for s in pipeline.steps])


## Test Data-ის ჩატვირთვა (raw, NOT preprocessed)

In [ ]:
KAGGLE_PATH = "/kaggle/input/ieee-fraud-detection"
LOCAL_PATH  = "./data"
DATA_PATH   = KAGGLE_PATH if os.path.exists(KAGGLE_PATH) else LOCAL_PATH

test_tx = pd.read_csv(f"{DATA_PATH}/test_transaction.csv")
test_id = pd.read_csv(f"{DATA_PATH}/test_identity.csv")

# Kaggle-ის test_identity-ში id-XX → id_XX (rename)
test_id.columns = [c.replace("id-", "id_") for c in test_id.columns]

test = test_tx.merge(test_id, on="TransactionID", how="left")
print("Test shape:", test.shape)
test_ids = test["TransactionID"].copy()


## იგივე Feature Engineering ვცადოთ test-ზე

Pipeline ცვლის preprocessing-ს, მაგრამ ჯერ უნდა დავამატოთ engineered ფიჩერები (იგივე რასაც training-ში დავამატეთ).

In [ ]:
test["TransactionAmt_log"] = np.log1p(test["TransactionAmt"])

d_cols = [c for c in test.columns if c.startswith("D") and c[1:].isdigit()]
c_cols = [c for c in test.columns if c.startswith("C") and c[1:].isdigit()]
m_cols = [c for c in test.columns if c.startswith("M") and c[1:].isdigit()]

if d_cols:
    test["D_mean"]       = test[d_cols].mean(axis=1)
    test["D_null_count"] = test[d_cols].isna().sum(axis=1)
if c_cols:
    test["C_sum"] = test[c_cols].fillna(0).sum(axis=1)
if m_cols:
    test["M_T_count"] = (test[m_cols] == "T").sum(axis=1)
    test["M_F_count"] = (test[m_cols] == "F").sum(axis=1)

print(f"Test დაემატა {len([c for c in ['TransactionAmt_log','D_mean','D_null_count','C_sum','M_T_count','M_F_count'] if c in test.columns])} ახალი ფიჩერი")


## Predict

Pipeline-ი თავად აპროცესორებს test set-ს. ჩვენ გვაძლევს fraud probability-ს.

In [ ]:
expected_cols = pipeline.feature_names_in_ if hasattr(pipeline, "feature_names_in_") else None
if expected_cols is not None:
    # ვამატებთ NaN-ების სვეტებს test-ში თუ რომელიმე აკლია
    missing = [c for c in expected_cols if c not in test.columns]
    for c in missing:
        test[c] = np.nan
    X_test = test[expected_cols]
else:
    X_test = test.drop(columns=["TransactionID"], errors="ignore")

predictions = pipeline.predict_proba(X_test)[:, 1]
print("Predictions shape:", predictions.shape)
print(f"საშუალო fraud probability: {predictions.mean():.4f}")
print(f"Predicted fraud rate (>0.5): {(predictions > 0.5).mean():.4f}")


In [ ]:
submission = pd.DataFrame({
    "TransactionID": test_ids,
    "isFraud":       predictions,
})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print(f"\nშენახულია submission.csv ({len(submission)} row)")
